In [3]:
import pandas as pd
from dotenv import load_dotenv

from pronostico_demanda_electrica.baselines import (
    dividir_entrenamiento_prueba,
    naive_estacional,
    promedio_ultimos_12,
)
from pronostico_demanda_electrica.metrics import evaluar_por_region, wape
from pronostico_demanda_electrica.queries import leer_demanda_mensual
from pronostico_demanda_electrica.storage import crear_engine

load_dotenv()
mensual = leer_demanda_mensual(crear_engine())
entrenamiento, prueba = dividir_entrenamiento_prueba(mensual)

print(
    "Entrenamiento:",
    entrenamiento["mes"].min().date(),
    "a",
    entrenamiento["mes"].max().date(),
    "-",
    len(entrenamiento),
    "filas",
)
print(
    "Prueba:",
    prueba["mes"].min().date(),
    "a",
    prueba["mes"].max().date(),
    "-",
    len(prueba),
    "filas",
)

Entrenamiento: 2022-01-01 a 2025-08-01 - 396 filas
Prueba: 2025-09-01 a 2026-08-01 - 108 filas


In [4]:
pronosticos = {
    "promedio_12": promedio_ultimos_12(entrenamiento, prueba),
    "naive_estacional": naive_estacional(entrenamiento, prueba),
}

comparacion = pd.concat(
    {
        nombre: evaluar_por_region(prueba, p)["wape_%"]
        for nombre, p in pronosticos.items()
    },
    axis=1,
)
comparacion.round(1)

,promedio_12,naive_estacional
region,,
BCA-BCA,19.6,5.7
BCS-BCA,19.0,10.2
SIN-CEN,2.9,2.7
SIN-NES,15.1,3.8
SIN-NOR,23.8,7.0
SIN-NTE,16.9,4.4
SIN-OCC,4.5,2.8
SIN-ORI,8.2,3.3
SIN-PEN,13.7,4.3


In [5]:
{nombre: round(wape(prueba["demanda_gwh"], p), 2) for nombre, p in pronosticos.items()}

{'promedio_12': 10.72, 'naive_estacional': 3.83}